In [1]:
# Cell 1: Imports and setup
import os
import json
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd

# Cell 2: Genre normalization setup
target_genres = [
    'rock', 'pop', 'jazz', 'electronic', 'blues', 'metal', 'hiphop', 'country',
    'folk', 'soul', 'punk', 'classical', 'reggae', 'rnb', 'indie', 'funk',
    'dance', 'latin', 'ambient', 'experimental', 'world'
]

explicit_map = {
    'pink': 'pop',
    'p!nk': 'pop',
    'beatles': 'rock',
    'the beatles': 'rock',
    'stones': 'rock',
    'the stones': 'rock',
    'alternative': 'rock',
    'emo': 'punk',
    'shoegaze': 'rock',
    'acoustic': 'folk',
    'instrumental': 'other',
    'soundtrack': 'other',
    'british': 'pop',
    '80s': 'pop',
    "80's": 'pop',
    '90s': 'rock',
    '90': 'rock',
    '70s': 'pop',
    '70': 'pop',
    'raphiphop': 'hiphop',
    'hip hop': 'hiphop',
    'rap': 'hiphop',
    'indie rock': 'rock',
    'indie': 'indie',
    'funksoulrnb': 'soul',
    'folkcountry': 'country',
    # Add more mappings as needed
}

def normalize_tag(tag):
    """Normalize a genre tag to one of our target genres"""
    if tag is None:
        return 'other'
    
    t = tag.strip().lower()
    
    # Check explicit mappings first
    if t in explicit_map:
        return explicit_map[t]
    
    # Check if it's already a target genre
    if t in target_genres:
        return t
    
    # Check if target genre is contained in the tag
    for genre in target_genres:
        if genre in t:
            return genre
    
    # Pattern-based matching
    if 'vocal' in t or 'female' in t or 'male' in t:
        return 'pop'
    if 'singer' in t or 'songwriter' in t:
        return 'folk'
    if 'electronica' in t or 'electro' in t:
        return 'electronic'
    if 'blues' in t:
        return 'blues'
    if 'funk' in t:
        return 'funk'
    if 'ambient' in t or 'chill' in t:
        return 'ambient'
    if 'latin' in t or 'bossa' in t or 'samba' in t or 'salsa' in t or 'reggaeton' in t:
        return 'latin'
    if 'jazz' in t:
        return 'jazz'
    if 'punk' in t:
        return 'punk'
    if 'metal' in t:
        return 'metal'
    if 'rock' in t:
        return 'rock'
    if 'pop' in t:
        return 'pop'
    if 'country' in t:
        return 'country'
    if 'folk' in t:
        return 'folk'
    if 'indie' in t:
        return 'indie'
    if 'experimental' in t:
        return 'experimental'
    if 'dance' in t:
        return 'dance'
    if 'reggae' in t:
        return 'reggae'
    if 'soul' in t:
        return 'soul'
    if 'rnb' in t or 'r&b' in t:
        return 'rnb'
    if 'hiphop' in t or 'hip-hop' in t or 'rap' in t:
        return 'hiphop'
    
    return 'other'

In [2]:
# Cell 3: Functional harmony processing
def functional_relabel(chord):
    """Extract functional harmony label from chord data"""
    func = chord['functional_harmony'].get('functional', '')
    alter = chord['functional_harmony'].get('alterations', '')
    has_54 = False
    if isinstance(alter, str):
        has_54 = '54' in alter
    elif isinstance(alter, list):
        has_54 = '54' in alter
    if has_54:
        return f"{func}"
    return func

In [3]:
# Cell 4: N-gram generation functions
def make_ngram(chords, n=3):
    """
    Create n-grams from chord sequence with specific filtering rules:
    - Trigrams (n=3): All 3 chords must be different (case-insensitive)
    - Tetragrams (n=4): At least 3 different chords required (case-insensitive)
    - Other n-grams: At least 3 different chords required
    
    Args:
        chords: List of chord labels
        n: Size of n-gram (3 for trigrams, 4 for tetragrams, etc.)
    """
    if len(chords) < n:
        return []
    
    ngrams = []
    
    # Step 1: Remove consecutive duplicates from the original sequence (case-sensitive)
    filtered_chords = []
    for i, chord in enumerate(chords):
        # Only add if it's different from the previous chord (no consecutive duplicates)
        if i == 0 or chord != chords[i-1]:
            filtered_chords.append(chord)
    
    # Step 2: Create n-grams and apply filtering based on n-gram size
    for i in range(len(filtered_chords) - n + 1):
        ngram = tuple(filtered_chords[i:i+n])
        
        # Convert to lowercase for case-insensitive comparison
        lowercase_ngram = [chord.lower() for chord in ngram]
        unique_chords = len(set(lowercase_ngram))
        
        # Apply different rules based on n-gram size
        if n == 3:
            # Trigrams: All 3 chords must be different
            if unique_chords == 3:
                ngrams.append(ngram)
        else:
            # Tetragrams and others: At least 3 different chords required
            if unique_chords >= 3:
                ngrams.append(ngram)
    
    return ngrams

def make_trigram(chords):
    """Wrapper for trigrams"""
    return make_ngram(chords, n=3)

def make_tetragram(chords):
    """Wrapper for tetragrams"""
    return make_ngram(chords, n=4)

In [4]:
# Cell 5: Style extraction functions
def extract_styles(style_json_path):
    """Extract and normalize styles from style JSON file"""
    with open(style_json_path, "r") as f:
        data = json.load(f)

    # Process original styles (human annotations)
    original = data.get("original", None)
    normalized_original_styles = []
    
    if isinstance(original, list):
        for style in original:
            normalized_style = normalize_tag(style)
            if normalized_style != 'other':  # Only keep meaningful genres
                normalized_original_styles.append(normalized_style)
    elif isinstance(original, str):
        styles = [s.strip() for s in original.split(",") if s.strip()]
        for style in styles:
            normalized_style = normalize_tag(style)
            if normalized_style != 'other':  # Only keep meaningful genres
                normalized_original_styles.append(normalized_style)
    
    # Process Dortmund genres (ML predictions)
    genres = data.get("genre_dortmund", {})
    dortmund_genres = []
    for genre, confidence in genres.items():
        dortmund_genres.append({
            'genre': normalize_tag(genre),
            'confidence': float(confidence)
        })
    
    return normalized_original_styles, dortmund_genres

In [5]:
# Cell 6: Harmony data extraction
def extract_ngrams_from_harmony(harmony_json_path, n=3):
    """Extract n-grams from harmony JSON file"""
    if not os.path.exists(harmony_json_path):
        return []
    
    try:
        with open(harmony_json_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Problem loading {harmony_json_path}: {e}")
        return []
    
    # Handle different data structures
    if isinstance(data, list) and len(data) > 0 and "functional_harmony" in data[0]:
        chords = [functional_relabel(chord) for chord in data]
    elif isinstance(data, dict) and "chords" in data:
        chords = [functional_relabel(chord) for chord in data["chords"]]
    else:
        chords = []
    
    # Generate n-grams
    return make_ngram(chords, n)

In [ ]:
# Cell 7: Main n-gram aggregation function
def create_ngram_aggregated_dataset(base_path="/workspace/dataset_corrected", 
                                   collections=["lastfm", "suno", "udio"],
                                   n=3):
    """
    Create n-gram aggregated dataset where each n-gram is a datapoint
    
    Args:
        base_path: Path to dataset collections
        collections: List of collection names
        n: Size of n-gram (3 for trigrams, 4 for tetragrams)
    
    Returns:
        Dictionary with n-gram statistics per collection
    """
    print(f"Creating aggregated {n}-gram dataset...")
    
    # Structure: {collection: {ngram_tuple: {count, styles, dortmund_genres, song_ids}}}
    collection_data = {}
    
    for collection in collections:
        print(f"Processing collection: {collection}")
        collection_path = os.path.join(base_path, collection)
        
        if not os.path.exists(collection_path):
            print(f"Warning: Collection path {collection_path} does not exist")
            continue
            
        # Initialize data structure for this collection
        ngram_data = defaultdict(lambda: {
            "count": 0,
            "human_styles": Counter(),  # Counter for human annotated styles
            "dortmund_genres": Counter(),  # Counter for Dortmund ML predictions (weighted)
            "song_ids": []  # List of song_ids where this n-gram appears
        })
        
        song_ids = [d for d in os.listdir(collection_path) 
                   if os.path.isdir(os.path.join(collection_path, d))]
        
        for song_id in tqdm(song_ids, desc=f"Processing {collection}"):
            song_path = os.path.join(collection_path, song_id)
            style_json = os.path.join(song_path, f"{song_id}_style.json")
            harmony_json = os.path.join(song_path, f"{song_id}_analysis.json") #_normalized.json
            
            # Skip if files don't exist
            if not (os.path.exists(style_json) and os.path.exists(harmony_json)):
                continue
            
            # Extract styles
            try:
                original_styles, dortmund_genres = extract_styles(style_json)
            except Exception as e:
                print(f"Error processing styles for {song_id}: {e}")
                continue
            
            # Extract n-grams
            ngrams = extract_ngrams_from_harmony(harmony_json, n)
            
            # Aggregate data for each n-gram found in this song
            for ngram in ngrams:
                ngram_key = tuple(ngram)
                
                # Increment count
                ngram_data[ngram_key]["count"] += 1
                
                # Add song_id if not already present
                if song_id not in ngram_data[ngram_key]["song_ids"]:
                    ngram_data[ngram_key]["song_ids"].append(song_id)
                
                # Add human styles
                for style in original_styles:
                    ngram_data[ngram_key]["human_styles"][style] += 1
                
                # Add Dortmund genres with their confidence weights
                for genre_info in dortmund_genres:
                    genre = genre_info['genre']
                    confidence = genre_info['confidence']
                    ngram_data[ngram_key]["dortmund_genres"][genre] += confidence
        
        collection_data[collection] = dict(ngram_data)
        print(f"  Found {len(ngram_data)} unique {n}-grams in {collection}")
    
    return collection_data

In [7]:
# Cell 8: Dataset export function
def export_ngram_dataset(collection_data, n, output_dir="/workspace/ngram_datasets"):
    """Export n-gram dataset to organized JSON files"""
    
    # Create output directories
    if n == 3:
        output_path = os.path.join(output_dir, "trigrams")
    elif n == 4:
        output_path = os.path.join(output_dir, "tetragrams")
    else:
        output_path = os.path.join(output_dir, f"{n}grams")
    
    os.makedirs(output_path, exist_ok=True)
    
    for collection, ngram_data in collection_data.items():
        # Convert to exportable format
        export_data = []
        
        for ngram_tuple, stats in ngram_data.items():
            # Get top 10 human styles
            top_human_styles = stats["human_styles"].most_common(10)
            
            # Get all Dortmund genres sorted by weight
            top_dortmund_genres = sorted(
                stats["dortmund_genres"].items(), 
                key=lambda x: x[1], 
                reverse=True
            )
            
            export_entry = {
                "ngram": list(ngram_tuple),  # Convert tuple to list for JSON
                "count": stats["count"],
                "human_styles": top_human_styles,  # [(style, count), ...]
                "dortmund_genres": top_dortmund_genres,  # [(genre, weight), ...]
                "song_ids": stats["song_ids"],
                "num_songs": len(stats["song_ids"])
            }
            export_data.append(export_entry)
        
        # Sort by count (most frequent first)
        export_data.sort(key=lambda x: x["count"], reverse=True)
        
        # Write to file
        output_file = os.path.join(output_path, f"{n}gram_dataset_{collection}.json")
        with open(output_file, 'w') as f:
            json.dump(export_data, f, indent=2)
        
        print(f"Exported {len(export_data)} {n}-grams for {collection} to {output_file}")

In [8]:
# Cell 9: Create datasets
# Configuration
BASE_PATH = "/workspace/dataset_corrected"
COLLECTIONS = ["lastfm", "suno", "udio"]

# Create trigram dataset
print("Creating trigram aggregated dataset...")
trigram_collection_data = create_ngram_aggregated_dataset(BASE_PATH, COLLECTIONS, n=3)

# Create tetragram dataset
print("\nCreating tetragram aggregated dataset...")
tetragram_collection_data = create_ngram_aggregated_dataset(BASE_PATH, COLLECTIONS, n=4)

# Cell 10: Export datasets
export_ngram_dataset(trigram_collection_data, n=3)
export_ngram_dataset(tetragram_collection_data, n=4)

# Cell 11: Print summary statistics
def print_collection_stats(collection_data, n):
    """Print statistics for each collection"""
    print(f"\n=== {n}-gram Collection Statistics ===")
    
    for collection, ngram_data in collection_data.items():
        total_ngrams = sum(stats["count"] for stats in ngram_data.values())
        unique_ngrams = len(ngram_data)
        
        print(f"\n{collection.upper()}:")
        print(f"  Unique {n}-grams: {unique_ngrams:,}")
        print(f"  Total {n}-gram instances: {total_ngrams:,}")
        
        # Find most common n-gram
        if ngram_data:
            most_common = max(ngram_data.items(), key=lambda x: x[1]["count"])
            ngram, stats = most_common
            print(f"  Most common {n}-gram: {' - '.join(ngram)} (count: {stats['count']})")
            
            # Show top human styles for most common n-gram
            if stats["human_styles"]:
                top_style = stats["human_styles"].most_common(1)[0]
                print(f"    Top style: {top_style[0]} ({top_style[1]} occurrences)")

print_collection_stats(trigram_collection_data, 3)
print_collection_stats(tetragram_collection_data, 4)

# Cell 12: Show final directory structure
print(f"\n📁 Files created:")
print(f"/workspace/ngram_datasets/")
print(f"├── 📁 trigrams/")
for collection in COLLECTIONS:
    print(f"│   ├── 📄 3gram_dataset_{collection}.json")
print(f"└── 📁 tetragrams/")
for collection in COLLECTIONS:
    print(f"    ├── 📄 4gram_dataset_{collection}.json")

Creating trigram aggregated dataset...
Creating aggregated 3-gram dataset...
Processing collection: lastfm


Processing lastfm: 100%|██████████| 19913/19913 [03:19<00:00, 99.99it/s] 


  Found 92098 unique 3-grams in lastfm
Processing collection: suno


Processing suno: 100%|██████████| 19972/19972 [02:06<00:00, 157.53it/s]


  Found 51951 unique 3-grams in suno
Processing collection: udio


...

  Found 578712 unique 4-grams in lastfm
Processing collection: suno


Processing suno: 100%|██████████| 19972/19972 [02:10<00:00, 153.01it/s]


  Found 256945 unique 4-grams in suno
Processing collection: udio


Processing udio: 100%|██████████| 19992/19992 [01:59<00:00, 166.67it/s]


  Found 374603 unique 4-grams in udio
Exported 92098 3-grams for lastfm to /workspace/ngram_datasets/trigrams/3gram_dataset_lastfm.json
Exported 51951 3-grams for suno to /workspace/ngram_datasets/trigrams/3gram_dataset_suno.json
Exported 67460 3-grams for udio to /workspace/ngram_datasets/trigrams/3gram_dataset_udio.json
Exported 578712 4-grams for lastfm to /workspace/ngram_datasets/tetragrams/4gram_dataset_lastfm.json
Exported 256945 4-grams for suno to /workspace/ngram_datasets/tetragrams/4gram_dataset_suno.json
Exported 374603 4-grams for udio to /workspace/ngram_datasets/tetragrams/4gram_dataset_udio.json

=== 3-gram Collection Statistics ===

LASTFM:
  Unique 3-grams: 92,098
  Total 3-gram instances: 2,342,883
  Most common 3-gram: IV - I - v (count: 6399)
    Top style: rock (12628 occurrences)

SUNO:
  Unique 3-grams: 51,951
  Total 3-gram instances: 1,483,050
  Most common 3-gram: IV - I - V (count: 16480)
    Top style: pop (9718 occurrences)

UDIO:
  Unique 3-grams: 67,460
